# Building with Managed Agents in the Gemini API

*A hands-on tutorial — from `hello world` to a custom agent defined in markdown.*

---

On May 19, 2026, Google [announced Managed Agents in the Gemini API](https://blog.google/innovation-and-ai/technology/developers-tools/managed-agents-gemini-api/) — a way to spin up an autonomous agent with a single API call. Each call provisions a fresh, isolated Linux sandbox where the agent can reason, write and execute code, manage files, and browse the web.

The pitch is that the boring-but-hard parts of shipping a production agent — sandboxing, orchestration, state management — are now handled for you. You bring the prompt and (optionally) a few markdown files that describe how your agent should behave.

In this notebook we'll walk through everything you need to actually use the feature:

1. A first interaction with the default **Antigravity** agent
2. Multi-turn conversations that keep files around between calls
3. Streaming
4. Pulling files out of the sandbox
5. Defining a **custom agent** with `AGENTS.md` and `SKILL.md`
6. An end-to-end mini-project: a weekly news digest agent

Heads up: at the time of writing, Managed Agents are in **public preview**. APIs can shift, and you should always review what an agent does before letting it touch anything sensitive.

## 0. Setup

You need:

- A Gemini API key — grab one at [aistudio.google.com/apikey](https://aistudio.google.com/apikey)
- The `google-genai` Python SDK and `python-dotenv` (installed in the cell above)

### Setting up your API key with `.env`

Create a file called `.env` in the same directory as this notebook:

```
GEMINI_API_KEY=AIza...your-real-key-here
```

Then add `.env` to your `.gitignore` so it never gets committed:

```bash
echo ".env" >> .gitignore
```

The next cell loads the key from `.env` into the environment, and the Gemini SDK picks it up automatically from there.

> ⚠️ **Two things before you run anything else.**
>
> **1. Don't paste the key into a notebook cell.** If you ever ran a cell that printed the key (even by accident), the value is now embedded in the saved `.ipynb` JSON even if you edited the cell afterward. Before committing this notebook: `Restart Kernel and Clear All Outputs`, then `grep AIza managed_agents_tutorial.ipynb` to confirm it's clean.
>
> **2. The sandbox is sandboxed. Your laptop is not.** This notebook later asks an agent to browse the web and pulls its output files to your local machine. We extract those files to a temp directory with an allowlist of expected names — but the contents themselves are still untrusted. Inspect before opening anything in a tool that auto-executes content (browsers, IDEs).

In [ ]:
%pip install -q --upgrade google-genai requests python-dotenv

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Load GEMINI_API_KEY from a local .env file. If you haven't created one,
# see the markdown cell above for instructions.
load_dotenv()

if not os.environ.get("GEMINI_API_KEY"):
    sys.exit(
        "GEMINI_API_KEY is not set.\n"
        "Create a .env file in this directory with one line:\n"
        "  GEMINI_API_KEY=AIza...your-real-key-here\n"
        "Then restart the kernel and re-run this cell.\n"
        "Do NOT paste the key into this cell — it will be saved into the notebook outputs."
    )

from google import genai
client = genai.Client()
print("Client ready.")

## 1. Your first agent call

The whole interface boils down to one method: `client.interactions.create(...)`.

Three parameters matter for a first call:

- `agent` — the agent ID. We'll use `"antigravity-preview-05-2026"`, Google's default general-purpose agent built on Gemini 3.5 Flash.
- `environment` — set to `"remote"` to provision a fresh sandbox.
- `input` — what you want the agent to do, in plain language.

Let's ask it to do something that actually exercises code execution and the filesystem.

In [ ]:
interaction = client.interactions.create(
    agent="antigravity-preview-05-2026",
    input=(
        "Write a Python script that generates the first 20 Fibonacci numbers "
        "and saves them to fibonacci.txt. Then read the file and print its contents."
    ),
    environment="remote",
)

print(f"Interaction ID : {interaction.id}")
print(f"Environment ID : {interaction.environment_id}")
print("--- Output ---")
print(interaction.output_text)

Two things to notice in the response:

- `interaction.id` — identifies this turn of the conversation. You'll use it to keep talking to the same agent.
- `interaction.environment_id` — identifies the sandbox. You'll use it to keep the same filesystem around.

If you want to inspect *what the agent did*, `interaction.steps` gives you the full trace: reasoning, tool calls, code execution, the lot.

In [ ]:
for i, step in enumerate(interaction.steps):
    print(f"Step {i}: {getattr(step, 'type', type(step).__name__)}")

## 2. Continuing the conversation

The API splits state into two independent dimensions:

| State | Controlled by | What it carries |
| --- | --- | --- |
| Conversation context | `previous_interaction_id` | chat history, reasoning trace, tool use |
| Environment state | `environment` | files, installed packages, sandbox state |

Passing both keeps the conversation *and* the workspace alive. The agent will remember the previous turn and `fibonacci.txt` will still be there.

In [ ]:
interaction_2 = client.interactions.create(
    agent="antigravity-preview-05-2026",
    previous_interaction_id=interaction.id,
    environment=interaction.environment_id,
    input="Now plot the Fibonacci sequence as a line chart and save it as chart.png.",
)

print(interaction_2.output_text)

You can mix and match the two state dimensions however you want:

- **Same workspace, new conversation** — omit `previous_interaction_id`, pass the environment ID. Useful when you want to start a new task but keep the files your agent already produced.
- **Same conversation, new workspace** — pass `previous_interaction_id`, set `environment="remote"`. Useful for swapping in a clean sandbox without losing chat history.

A small but lovely detail: at around 135k tokens, the managed agent runs an automatic context compaction step. You don't have to think about pruning history or fighting context rot — it does that on its own.

## 3. Streaming

For longer tasks — anything involving web browsing or a few tool-call hops — you'll want to stream so you can watch the agent work in real time. Same call, just add `stream=True` and iterate.

In [ ]:
stream = client.interactions.create(
    agent="antigravity-preview-05-2026",
    input="Read Hacker News, summarize the top 5 stories, and save the results as a PDF.",
    environment="remote",
    stream=True,
)

for event in stream:
    # Each event is a step delta: incremental text, reasoning, or tool-call update.
    print(event)

## 4. Pulling files out of the sandbox

The agent created `fibonacci.txt`, `chart.png`, and a PDF — but they all live inside the sandbox VM. To get them locally, hit the Files API with the environment ID and download a tarball of the workspace.

### ⚠️ Security note before downloading sandbox files

The agent ran code that touched the open web (or will, in later cells). Anything in the tarball — file names, paths, contents, link targets — was influenced by data the agent read from sources you didn't pick.

Three specific risks if you extract this carelessly:

1. **Tar slip / path traversal.** A malicious entry like `../../etc/something` can write outside the extraction directory.
2. **Symlink and hardlink trickery.** An entry can claim to be `report.pdf` but point at `~/.ssh/id_rsa` — extracting it either reads that file or overwrites it. Filename-only checks miss this entirely.
3. **Files landing somewhere your host trusts.** If you extract into your project root, your IDE indexes it, your shell sees it, your `.gitignore` may not cover it. A rogue `requirements.txt` or `.bashrc` becomes a foothold.

The pattern below applies four defenses together:

- **Allowlist by name** — only files you explicitly asked the agent to produce get extracted; everything else is logged and ignored.
- **Member-type check** — `member.isfile()` rejects symlinks, hardlinks, and device nodes, so the "named correctly but linked elsewhere" attack fails.
- **`filter="data"`** (Python 3.12+) — rejects path-traversal entries and absolute paths as a defense-in-depth layer.
- **Extract to a temp directory** — `tempfile.mkdtemp()` keeps the extraction off your project root. Move files back manually after inspection.

Edit `EXPECTED` in each cell to match what you asked the agent to produce.

In [ ]:
import os, requests, tarfile, tempfile

env_id = interaction.environment_id

response = requests.get(
    f"https://generativelanguage.googleapis.com/v1beta/files/environment-{env_id}:download",
    params={"alt": "media"},
    headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]},
    allow_redirects=True,
)
response.raise_for_status()

work_dir = tempfile.mkdtemp(prefix="sandbox_snapshot_")
tar_path = os.path.join(work_dir, "snapshot.tar")
with open(tar_path, "wb") as f:
    f.write(response.content)

# ─── EDIT THIS PER USE CASE ─────────────────────────────────────────────
EXPECTED = {"fibonacci.txt", "chart.png"}
# ────────────────────────────────────────────────────────────────────────

def normalize(name: str) -> str:
    """Strip './' and leading slashes so 'fibonacci.txt' matches './fibonacci.txt'."""
    return name.lstrip("./").lstrip("/")

with tarfile.open(tar_path) as tar:
    # Build candidates by normalized name. Sandbox tarballs store files as
    # './fibonacci.txt', not 'fibonacci.txt' — without normalize() you get
    # 'missing: fibonacci.txt' for files that are right there.
    candidates = {}
    print("Top-level files in archive:")
    for m in tar.getmembers():
        norm = normalize(m.name)
        # Show only top-level files — the snapshot includes everything the
        # agent installed (matplotlib, Pillow, fontTools…), which is thousands
        # of files we don't want to drown the output in.
        if "/" not in norm and m.isfile():
            marker = "✓" if norm in EXPECTED else "·"
            print(f"  {marker} {m.name}  ({m.size} bytes)")
        if norm in EXPECTED and m.isfile():
            candidates[norm] = m

    print()
    for name in EXPECTED:
        if name not in candidates:
            print(f"  missing: {name}")
            continue
        member = candidates[name]
        # Reject anything that isn't a plain regular file (the isfile() check
        # is already enforced above by the candidates filter; this is here
        # as documentation of intent).
        if not member.isfile():
            print(f"  refused (not a regular file): {name}")
            continue
        # Rewrite the path so it extracts as 'fibonacci.txt', not './fibonacci.txt'.
        member.name = name
        # filter='data' is defense in depth — rejects path traversal even
        # though we picked the name explicitly.
        tar.extract(member, path=work_dir, filter="data")
        print(f"  extracted: {os.path.join(work_dir, name)}")

print(f"\nWork dir: {work_dir}")
print("Inspect contents before copying anything back to your project.")

> **Note on the pure-Python pattern:** the cell above already uses Python's `tarfile` module — there's no shell-vs-Python split anymore. The four defenses (allowlist + member-type check + `filter="data"` + temp dir) compose into one pattern that works the same on macOS, Linux, and Windows. The cell above is the reference implementation.

## 5. Defining your own custom agent

Here's the part that makes this feature interesting. Instead of stuffing a giant system prompt into every call, you can define an agent declaratively:

- An **`AGENTS.md`** file with the agent's instructions, role, and behavior
- Optional **`SKILL.md`** files that bundle reusable capabilities (a procedure, a tool wrapper, a domain heuristic)

These files live in the sandbox under `.agents/`, and the managed agent loads them automatically. They're plain markdown, so you can version them, code-review them, and share them across teams.

There are two ways to set up the base environment for a custom agent:

**Option A — Inline sources or a Git repo.** Define files directly or pull a skills directory from GitHub:

In [ ]:
agent = client.agents.create(
    id="fibonacci-analyst",
    base_agent="antigravity-preview-05-2026",
    system_instruction=(
        "You are a math analysis agent. Generate sequences, visualize them, "
        "and export results as PDF reports."
    ),
    base_environment={
        "type": "remote",
        "sources": [
            {
                "type": "inline",
                "target": ".agents/AGENTS.md",
                "content": "Always include a chart and a summary table in your reports.",
            },
            # To pull skills from a real git repo, add:
            # {
            #     "type": "repository",
            #     "source": "https://github.com/your-org/your-real-repo",
            #     "target": ".agents/skills",
            # },
            # Important: if the repo URL is unreachable, agents.create() succeeds
            # but the agent is created in a broken state, and the next
            # interactions.create() call returns a generic "400 invalid_request"
            # with no hint that the repo was the problem. Use inline sources
            # while prototyping; only add `repository` once you have a real URL.
        ],
    },
)

print(f"Created agent: {agent.id}")

**Option B — From an existing environment.** If you've already poked at a sandbox interactively and gotten it to a useful state (installed packages, cached data, scratch files), promote it as-is:

In [ ]:
agent = client.agents.create(
    id="fibonacci-analyst-from-env",
    base_agent="antigravity-preview-05-2026",
    system_instruction="You are a math analysis agent.",
    base_environment=interaction.environment_id,
)

## 6. Invoking your custom agent

Once an agent is registered, you call it by ID and the system instruction, tools, and base environment are applied automatically. Each interaction *forks* the base environment, so every run starts from the same clean state.

In [ ]:
result = client.interactions.create(
    agent="fibonacci-analyst",
    input="Generate the first 50 prime numbers, plot their distribution, and save a PDF report.",
    environment="remote",
)

print(result.output_text)

Any of the registered config — system instruction, tools, environment — can still be overridden per-interaction when you need a one-off variation.

## 7. Mini-project: a weekly news digest agent

Let's tie everything together with something close to a real use case: an agent that browses the web for stories on a topic, writes a digest, and saves it as a styled PDF.

We'll do this in three parts:

1. Write the `AGENTS.md` and a `SKILL.md` inline
2. Register the custom agent
3. Run it for a topic

### 7a. The agent definition

We'll keep it short — these are the kinds of files you'd version in a repo.

In [ ]:
AGENTS_MD = """\
# News Digest Agent

You are a research analyst that produces concise weekly news digests on a topic
the user provides.

## Process
1. Browse the web for 5–8 recent, high-quality stories on the topic.
2. Prefer original reporting and primary sources over aggregators.
3. For each story, extract: title, source, one-paragraph summary, and a link.
4. Group stories into 2–4 themes that emerge across the stories.
5. Produce a PDF report named `digest.pdf` using the `pdf_report` skill.

## Style
- Neutral, factual tone. No editorializing.
- Keep summaries under 80 words each.
- Lead the report with a 3-bullet TL;DR.
"""

SKILL_MD = """\
# Skill: pdf_report

Render a structured digest as a clean, readable PDF.

## When to use
Use this skill whenever the user wants the final deliverable as a PDF report.

## How
- Use the `fpdf2` Python library. If it is not installed, install it with
  `pip install --break-system-packages fpdf2` (the sandbox ships without
  most plotting/PDF libraries by default).
- Use a single column, 11pt body, 14pt section headers.
- First page: title, date, TL;DR bullets.
- Then one section per theme; under each theme list the stories with title,
  source, summary, and link.
- Save to `digest.pdf` at the workspace root.
"""

### 7b. Register the agent

In [ ]:
digest_agent = client.agents.create(
    id="weekly-news-digest",
    base_agent="antigravity-preview-05-2026",
    system_instruction="Follow the instructions in .agents/AGENTS.md and use the skills under .agents/skills/.",
    base_environment={
        "type": "remote",
        "sources": [
            {"type": "inline", "target": ".agents/AGENTS.md", "content": AGENTS_MD},
            {"type": "inline", "target": ".agents/skills/pdf_report/SKILL.md", "content": SKILL_MD},
        ],
    },
)

print(f"Created: {digest_agent.id}")

### 7c. Run it

In [ ]:
digest = client.interactions.create(
    agent="weekly-news-digest",
    input="Topic: open-source AI agent frameworks. Cover the last 7 days.",
    environment="remote",
)

print(digest.output_text)
print(f"\nEnvironment: {digest.environment_id}")

### 7d. Grab the PDF

Same download pattern as before, with the new environment ID.

In [ ]:
import os, requests, tarfile, tempfile

env_id = digest.environment_id

response = requests.get(
    f"https://generativelanguage.googleapis.com/v1beta/files/environment-{env_id}:download",
    params={"alt": "media"},
    headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]},
    allow_redirects=True,
)
response.raise_for_status()

work_dir = tempfile.mkdtemp(prefix="digest_snapshot_")
tar_path = os.path.join(work_dir, "snapshot.tar")
with open(tar_path, "wb") as f:
    f.write(response.content)

# ─── EDIT THIS PER USE CASE ─────────────────────────────────────────────
EXPECTED = {"digest.pdf"}
# ────────────────────────────────────────────────────────────────────────

def normalize(name: str) -> str:
    """Strip './' and leading slashes so 'digest.pdf' matches './digest.pdf'."""
    return name.lstrip("./").lstrip("/")

with tarfile.open(tar_path) as tar:
    candidates = {}
    print("Top-level files in archive:")
    for m in tar.getmembers():
        norm = normalize(m.name)
        if "/" not in norm and m.isfile():
            marker = "✓" if norm in EXPECTED else "·"
            print(f"  {marker} {m.name}  ({m.size} bytes)")
        if norm in EXPECTED and m.isfile():
            candidates[norm] = m

    print()
    for name in EXPECTED:
        if name not in candidates:
            print(f"  missing: {name}")
            continue
        member = candidates[name]
        if not member.isfile():
            print(f"  refused (not a regular file): {name}")
            continue
        member.name = name
        tar.extract(member, path=work_dir, filter="data")
        print(f"  extracted: {os.path.join(work_dir, name)}")

print(f"\nWork dir: {work_dir}")
print("The extracted PDF was produced by an agent that read untrusted web content.")
print("Open it in a sandboxed viewer; don't trust its links or embedded scripts.")

## Locking down the network

By default, Managed Agent environments have **unrestricted outbound network access**. That's convenient for prototyping — the agent can `pip install`, browse the web, hit any API — but too permissive for many production use cases. An agent that just read attacker-controlled content from a web page and has wildcard network access can exfiltrate anything it picks up to any host on the open internet.

You have three options for tightening this:

1. **Lock the network down completely** with `"network": "disabled"`. Use this for agents that only need to operate on local files and don't make outbound calls.
2. **Allowlist specific domains** with a `network.allowlist` block inside `base_environment`. Each entry has a `domain` (use `*` to match everything, or e.g. `*.example.com` for subdomains — note that `*.example.com` doesn't match `example.com` itself, you have to list both).
3. **Inject credentials** via `transform` headers on an allowlist entry. The token is added to outbound requests to that domain by an egress proxy and is never exposed inside the sandbox as an environment variable or file.

The cell below shows both patterns — a scoped allowlist for an issue-resolver agent and a fully-disabled-network agent for local-only work. Don't run it unless you actually want to create those agents.

In [ ]:
# Pattern 1: scoped allowlist — agent can reach GitHub and PyPI only.
# scoped_agent = client.agents.create(
#     id="scoped-issue-resolver",
#     base_agent="antigravity-preview-05-2026",
#     system_instruction="You triage GitHub issues. Only access github.com and pypi.org.",
#     base_environment={
#         "type": "remote",
#         "sources": [
#             {"type": "inline", "target": ".agents/AGENTS.md",
#              "content": "Use the GitHub API to read issues. Use pip for installs."},
#         ],
#         "network": {
#             "allowlist": [
#                 {"domain": "github.com"},
#                 {"domain": "api.github.com"},
#                 {"domain": "pypi.org"},
#                 {"domain": "files.pythonhosted.org"},  # PyPI downloads land here
#             ],
#         },
#     },
# )

# Pattern 2: fully isolated — agent has no outbound network at all.
# Use this when the agent only needs to operate on files you've already mounted.
# isolated_agent = client.agents.create(
#     id="local-only-analyst",
#     base_agent="antigravity-preview-05-2026",
#     system_instruction="Analyze local files only. No external calls.",
#     base_environment={
#         "type": "remote",
#         "network": "disabled",
#     },
# )

# What the wildcard does and why you should think before reaching for it:
#
#   "network": {"allowlist": [{"domain": "*"}]}
#
# The wildcard means the agent can reach any host on the public internet —
# which is the same as the default. Use a wildcard when you actually need the
# open web (browsing, search, package mirrors). Use named domains when you
# don't. Wildcards don't pair well with sandboxes that read untrusted input,
# because a prompt injection on a web page becomes "agent posts your scraped
# data to attacker-controlled URL."

print("Network is OPEN by default. Restrict it with network.allowlist, or block it entirely with network=\"disabled\".")

## 8. What to keep in mind

A few things worth knowing before you put this in front of users.

**Pricing.** Pay-as-you-go based on Gemini model tokens and tool usage. A single interaction can fire multiple reasoning loops and typically consumes anywhere from 100k to 3M tokens. Sandbox compute is *not* billed during the preview — that won't stay free forever.

**Limits.**
- Environments are deleted after 7 days of inactivity.
- VMs spin down when idle and cold-start on the next request — your files stick around, but expect added latency on the first call after a pause.
- Up to 1,000 managed agents per project.
- Sandbox base image: Ubuntu, Python 3.12, Node.js 22.

**Network access.** By default the sandbox has unrestricted outbound network. If your agent doesn't need that, lock it down with a network allowlist — this is one of those defaults you almost certainly want to tighten in production.

**Credentials.** External API keys are injected via egress proxy header transformations, so they're never exposed inside the sandbox. Still: assume the agent will use any credential you grant it to the full extent of its scope. Use short-lived, least-privilege tokens.

**Human oversight.** Preview product. Read the steps. Don't wire it up to anything destructive without review.

## Where to go next

- [Agents Overview](https://ai.google.dev/gemini-api/docs/agents) — the docs hub
- [Building Custom Agents](https://ai.google.dev/gemini-api/docs/custom-agents) — full `AGENTS.md` / `SKILL.md` spec
- [Agent Environments](https://ai.google.dev/gemini-api/docs/agent-environment) — sandbox configuration, network rules, credentials
- [Antigravity Agent](https://ai.google.dev/gemini-api/docs/antigravity-agent) — capabilities and pricing for the default agent
- [AI Studio Playground](https://ai.dev/managed-agents) — try it without writing code first